# Controller Design (Milestone 2)

17-state LQR controller with actuator delay (tau_x, tau_y) and integral action (e_x, e_y, e_z), copied from `crazyflie-client_cook/labs/lab06/01c-analysis-with-integral-action.ipynb`.

With m = 0.0340 kg and the Q below, this notebook reproduces `gains_flight20.json` (the gains flown in Labs 7-8) exactly.

## 1. Set up the notebook

### 1.1 Do imports

Do imports.

In [1]:
import numpy as np
import sympy as sym
from scipy import linalg
import matplotlib.pyplot as plt
from ae483.tools import *
from ae483.mytools import *
from ae483.design import *

### 1.2 Create an LQR solver

Define a function that solves the linear quadratic regulator (LQR) problem.

In [2]:
def lqr(A, B, Q, R):
    P = linalg.solve_continuous_are(A, B, Q, R)
    K = linalg.inv(R) @  B.T @ P
    return K

## 2. Derive equations of motion

### 2.1 Define symbolic variables

Define states.

In [3]:
# components of position (meters)
p_x, p_y, p_z = sym.symbols('p_x, p_y, p_z')

# yaw, pitch, and roll angles (radians)
psi, theta, phi = sym.symbols('psi, theta, phi')

# components of linear velocity (meters / second)
v_x, v_y, v_z = sym.symbols('v_x, v_y, v_z')

# components of angular velocity (radians / second)
w_x, w_y, w_z = sym.symbols('w_x, w_y, w_z')

# integrals of position error (meter-seconds)
e_x, e_y, e_z = sym.symbols('e_x, e_y, e_z')

Define inputs.

In [4]:
# components of net rotor torque
tau_x, tau_y, tau_z = sym.symbols('tau_x, tau_y, tau_z')

# net rotor force
f_z = sym.symbols('f_z')

# commanded net torques
tau_x_cmd, tau_y_cmd = sym.symbols('tau_x_cmd, tau_y_cmd')

Define parameters.

In [5]:
m, J_x, J_y, J_z, g = sym.symbols('m, J_x, J_y, J_z, g')

# time constant of the motor response (seconds)
T_F = sym.symbols('T_F')

Create the linear velocity vector $v^B_{W, B}$ and the angular velocity vector $w^B_{W, B}$, both written in the coordinates of the body frame.

In [6]:
v_inB_ofWB = sym.Matrix([v_x, v_y, v_z])
w_inB_ofWB = sym.Matrix([w_x, w_y, w_z])

Create moment of inertia matrix (in coordinates of the body frame).

In [7]:
J_inB = sym.diag(J_x, J_y, J_z)

### 2.2 Define kinematics of orientation

#### 2.2.1 Rotation matrix in terms of yaw, pitch, roll angles

Define individual rotation matrices.

In [8]:
Rz = sym.Matrix([[sym.cos(psi), -sym.sin(psi), 0],
                 [sym.sin(psi), sym.cos(psi), 0],
                 [0, 0, 1]])

Ry = sym.Matrix([[sym.cos(theta), 0, sym.sin(theta)],
                 [0, 1, 0],
                 [-sym.sin(theta), 0, sym.cos(theta)]])

Rx = sym.Matrix([[1, 0, 0],
                 [0, sym.cos(phi), -sym.sin(phi)],
                 [0, sym.sin(phi), sym.cos(phi)]])

Apply sequential transformation to compute the rotation matrix that describes the orientation of the drone (i.e., of frame $B$ in the coordinates of frame $W$).

In [9]:
R_inW_ofB = Rz @ Ry @ Rx

#### 2.2.2 Map from angular velocity to angular rates

Recall that

$$\begin{bmatrix} \dot{\psi} \\ \dot{\theta} \\ \dot{\phi} \end{bmatrix} = N w_{W, B}^{B}$$

for some matrix $N$. Here is how to compute that matrix for a ZYX (yaw, pitch, roll) Euler angle sequence.  First, we compute its inverse:

In [10]:
Ninv = sym.Matrix.hstack((Ry @ Rx).T @ sym.Matrix([0, 0, 1]),
                              (Rx).T @ sym.Matrix([0, 1, 0]),
                                       sym.Matrix([1, 0, 0]))

Then, we compute $N$ by taking the inverse of $N^{-1}$:

In [11]:
N = sym.simplify(Ninv.inv())

### 2.3 Define equations of motion

Forces.

In [12]:
f_inB = R_inW_ofB.T @ sym.Matrix([0, 0, -m * g]) + sym.Matrix([0, 0, f_z])

Torques.

In [13]:
tau_inB = sym.Matrix([tau_x, tau_y, tau_z])

Create equations of motion for the delayed response of the actual net torques $\tau_x$ and $\tau_y$ to the commanded net torques $\tau_{x,\text{cmd}}$ and $\tau_{y,\text{cmd}}$:

$$\dot{\tau}_x = (1/T_F)(\tau_{x,\text{cmd}} - \tau_x) \qquad \dot{\tau}_y = (1/T_F)(\tau_{y,\text{cmd}} - \tau_y)$$

In [14]:
tau_dot = sym.Matrix([
    (1 / T_F) * (tau_x_cmd - tau_x),
    (1 / T_F) * (tau_y_cmd - tau_y),
])

Create equations of motion.

In [15]:
f_sym = sym.Matrix.vstack(
    R_inW_ofB @ v_inB_ofWB,
    N @ w_inB_ofWB,
    (1 / m) * (f_inB - w_inB_ofWB.cross(m * v_inB_ofWB)),
    J_inB.inv() @ (tau_inB - w_inB_ofWB.cross(J_inB @ w_inB_ofWB)),
    tau_dot,
    sym.Matrix([p_x, p_y, p_z]),
)

Show the right-hand side of the equations of motion, which have the form

$$\dot{s} = f(s, i, p)$$

for states

$$
s = \begin{bmatrix} p_x \\ p_y \\ p_z \\ \psi \\ \theta \\ \phi \\ v_x \\ v_y \\ v_z \\ w_x \\ w_y \\ w_z \\ \tau_x \\ \tau_y \\ e_x \\ e_y \\ e_z \end{bmatrix},
$$

inputs
$$
i = \begin{bmatrix} \tau_{x,\text{cmd}} \\ \tau_{y,\text{cmd}} \\ \tau_z \\ f_z \end{bmatrix},
$$

and parameters
$$
p = \begin{bmatrix} m \\ J_x \\ J_y \\ J_z \\ g \\ T_F \end{bmatrix}.
$$

In [16]:
f_sym

Matrix([
[ v_x*cos(psi)*cos(theta) + v_y*(sin(phi)*sin(theta)*cos(psi) - sin(psi)*cos(phi)) + v_z*(sin(phi)*sin(psi) + sin(theta)*cos(phi)*cos(psi))],
[v_x*sin(psi)*cos(theta) + v_y*(sin(phi)*sin(psi)*sin(theta) + cos(phi)*cos(psi)) + v_z*(-sin(phi)*cos(psi) + sin(psi)*sin(theta)*cos(phi))],
[                                                                       -v_x*sin(theta) + v_y*sin(phi)*cos(theta) + v_z*cos(phi)*cos(theta)],
[                                                                                         w_y*sin(phi)/cos(theta) + w_z*cos(phi)/cos(theta)],
[                                                                                                               w_y*cos(phi) - w_z*sin(phi)],
[                                                                                   w_x + w_y*sin(phi)*tan(theta) + w_z*cos(phi)*tan(theta)],
[                                                                                                (g*m*sin(theta) + m*v_y*w_z - m*v_z*w_y)/m

## 3. Derive state-space model

### 3.1 Choose equilibrium point

An equilibrium point of the nonlinear system is a choice of states $s_\text{eq}$ and inputs $i_\text{eq}$ — along with constant parameters $p_\text{eq}$ — for which

$$0 = f(s_\text{eq}, i_\text{eq}, p_\text{eq}).$$

Create a list of states, inputs, and parameters as symbolic variables.

In [17]:
s = [p_x, p_y, p_z, psi, theta, phi, v_x, v_y, v_z, w_x, w_y, w_z, tau_x, tau_y, e_x, e_y, e_z]
i = [tau_x_cmd, tau_y_cmd, tau_z, f_z]
p = [m, J_x, J_y, J_z, g, T_F]

Create a list of states to track as symbolic variables. These are states whose desired values will be specified by a client.

In [18]:
s_with_des = [p_x, p_y, p_z]

Create a function that evaluates $f(\cdot)$ at particular values of $s$, $i$, and $p$.

In [19]:
f = sym.lambdify(s + i + p, f_sym)

Define constants.

In [20]:
# Mass
# - 0.0340 kg: bare drone (gains_flight20.json)
# - 0.0390 kg: drone + empty box (Task 5, measured)
m = 0.0390

# Principle moments of inertia
# Box v1 measured (project-MOI/runs/box_v1)
J_x = 1.846e-05
J_y = 1.899e-05
J_z = 3.144e-05


# Acceleration of gravity
g = 9.81

# Time constant of the motor response (from lab04)
T_F = 0.056

Create a list of parameter values in the **same order** as the symbolic list. These are the parameter estimates we found in our experiments. They are not choices. (We use the subscript `_eq` to be consistent with what follows, and could say "parameter values *at equilibrium*," but don't be misled. These parameter values are *given* and are *constant* - again, they aren't choices.)

In [21]:
p_eq = [m, J_x, J_y, J_z, g, T_F]

Create a list of state and input values at equilibrium in the **same order** as the symbolic lists.

In [22]:
s_eq = [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]
i_eq = [0., 0., 0., m * g]


Evaluate the equations of motion at the equilibrium point — if it actually *is* an equilibrium point, then the result should be an array of zeros:

In [23]:
print(f(*s_eq, *i_eq, *p_eq))
assert(np.allclose(f(*s_eq, *i_eq, *p_eq), 0.))

[[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]


Note that this equilibrium point would remain an equilibrium point for any choice of `p_x`, `p_y`, and `p_z` — that is one thing that allows the controller to track desired values of these variables.

### 3.2 Compute A and B

We want to find

$$
A = \frac{\partial f}{\partial s}\biggr\vert_{(s, i, p) = (s_\text{eq}, i_\text{eq}, p_\text{eq})}
\qquad\text{and}\qquad
B = \frac{\partial f}{\partial i}\biggr\vert_{(s, i, p) = (s_\text{eq}, i_\text{eq}, p_\text{eq})}.
$$

First, we compute each Jacobian (i.e., each matrix of partial derivatives) in symbolic form.

In [24]:
A_sym = f_sym.jacobian(s)
B_sym = f_sym.jacobian(i)

Then, we create functions that allow us to evaluate these Jacobians at particular values of $s$, $i$, and $p$.

In [25]:
A_num = sym.lambdify(s + i + p, A_sym)
B_num = sym.lambdify(s + i + p, B_sym)

Finally, we plug in our equilibrium point.

In [26]:
A = A_num(*s_eq, *i_eq, *p_eq)
B = B_num(*s_eq, *i_eq, *p_eq)

Show $A$ (formatted nicely).

In [27]:
A_str = np.array2string(A,
                        formatter={'float_kind': lambda x: f'{x:6.3f}'},
                        prefix='    ',
                        max_line_width=np.inf)

print(f'A = {A_str}')

A = [[ 0.000  0.000  0.000  0.000  0.000  0.000  1.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000]
     [ 0.000  0.000  0.000  0.000  0.000  0.000  0.000  1.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000]
     [ 0.000  0.000  0.000  0.000 -0.000  0.000 -0.000  0.000  1.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000]
     [ 0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  1.000  0.000  0.000  0.000  0.000  0.000]
     [ 0.000  0.000  0.000  0.000  0.000 -0.000  0.000  0.000  0.000  0.000  1.000 -0.000  0.000  0.000  0.000  0.000  0.000]
     [ 0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  1.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000]
     [ 0.000  0.000  0.000  0.000  9.810  0.000  0.000  0.000 -0.000  0.000 -0.000  0.000  0.000  0.000  0.000  0.000  0.000]
     [ 0.000  0.000  0.000  0.000  0.000 -9.810 -0.000  0.000  0.000  0.000  0.000 -0.000  0.000  0.000  0.000  0.000 

Show $B$ (formatted nicely).

In [28]:
B_str = np.array2string(B,
                        formatter={'float_kind': lambda x: f'{x:11.3f}'},
                        prefix='    ',
                        max_line_width=np.inf)

print(f'B = {B_str}')

B = [[      0.000       0.000       0.000       0.000]
     [      0.000       0.000       0.000       0.000]
     [      0.000       0.000       0.000       0.000]
     [      0.000       0.000       0.000       0.000]
     [      0.000       0.000       0.000       0.000]
     [      0.000       0.000       0.000       0.000]
     [      0.000       0.000       0.000       0.000]
     [      0.000       0.000       0.000       0.000]
     [      0.000       0.000       0.000      25.641]
     [      0.000       0.000       0.000       0.000]
     [      0.000       0.000       0.000       0.000]
     [      0.000       0.000   31806.616       0.000]
     [     17.857       0.000       0.000       0.000]
     [      0.000      17.857       0.000       0.000]
     [      0.000       0.000       0.000       0.000]
     [      0.000       0.000       0.000       0.000]
     [      0.000       0.000       0.000       0.000]]


The state-space system is described by

$$ \dot{x} = Ax + Bu $$

where

$$ x = s - s_\text{eq} $$

and

$$ u = i - i_\text{eq}. $$

Note that $A$ and $B$ would remain the same for any choice of `p_x`, `p_y`, and `p_z` — that is another thing that allows the controller to track desired values of these variables.

## 4. Design method of power distribution

Define constants.

In [29]:
k_F = 1.4932e-06
k_M = 5.1759e-09
l = .0325 

Define the matrix $P$ that maps motor power commands ($m_1$, $m_2$, $m_3$, $m_4$) to inputs ($\tau_x$, $\tau_y$, $\tau_z$, $f_z$).

In [30]:
P = np.array([[ -l * k_F, -l * k_F,  l * k_F,  l * k_F  ],
              [ -l * k_F, l * k_F,   l * k_F,  -l * k_F ],
              [ -k_M,     k_M,       -k_M,     k_M      ],
              [ k_F,      k_F,       k_F,      k_F      ]])

Compute the matrix $P^{-1}$ that maps inputs to motor power commands.

In [31]:
Pinv = linalg.inv(P)

Show the matrix $P^{-1}$ (formatted nicely).

In [32]:
Pinv_str = np.array2string(Pinv,
                           formatter={'float_kind': lambda x: f'{x:12.1f}'},
                           prefix='         ',
                           max_line_width=np.inf)

print(f'inv(P) = {Pinv_str}')

inv(P) = [[  -5151558.9   -5151558.9  -48300778.6     167425.7]
          [  -5151558.9    5151558.9   48300778.6     167425.7]
          [   5151558.9    5151558.9  -48300778.6     167425.7]
          [   5151558.9   -5151558.9   48300778.6     167425.7]]


Print code that implements the method of power distribution in C (compare this code to $P^{-1}$).

In [33]:
export_power_distribution(
    Pinv,
    i=i,
    limiter='limitUint16',
    suffix='f',
    line_ending=';',
)

m_1 = limitUint16( -5151558.9f * tau_x_cmd -5151558.9f * tau_y_cmd -48300778.6f * tau_z + 167425.7f * f_z );
m_2 = limitUint16( -5151558.9f * tau_x_cmd + 5151558.9f * tau_y_cmd + 48300778.6f * tau_z + 167425.7f * f_z );
m_3 = limitUint16( 5151558.9f * tau_x_cmd + 5151558.9f * tau_y_cmd -48300778.6f * tau_z + 167425.7f * f_z );
m_4 = limitUint16( 5151558.9f * tau_x_cmd -5151558.9f * tau_y_cmd + 48300778.6f * tau_z + 167425.7f * f_z );


## 5. Design, implement, and test a sequence of controllers

Decide which inputs will depend on which states — we call this the **gain structure**.

In [34]:
# Find the gain structure
structure = gain_structure(A, B, s, i)

# Show the gain structure
structure

{'tau_x_cmd': ['p_y', 'phi', 'v_y', 'w_x', 'tau_x', 'e_y'],
 'tau_y_cmd': ['p_x', 'theta', 'v_x', 'w_y', 'tau_y', 'e_x'],
 'tau_z': ['psi', 'w_z'],
 'f_z': ['p_z', 'v_z', 'e_z']}

Print code that defines all gains in C, with placeholder values.

In [35]:
export_declarations(structure)

// Gains (these are set by flight.py - do not edit the values here)
static float eq_tau_x_cmd = 0.0f;
static float k_tau_x_cmd_p_y = 0.0f;
static float k_tau_x_cmd_phi = 0.0f;
static float k_tau_x_cmd_v_y = 0.0f;
static float k_tau_x_cmd_w_x = 0.0f;
static float k_tau_x_cmd_tau_x = 0.0f;
static float k_tau_x_cmd_e_y = 0.0f;
static float eq_tau_y_cmd = 0.0f;
static float k_tau_y_cmd_p_x = 0.0f;
static float k_tau_y_cmd_theta = 0.0f;
static float k_tau_y_cmd_v_x = 0.0f;
static float k_tau_y_cmd_w_y = 0.0f;
static float k_tau_y_cmd_tau_y = 0.0f;
static float k_tau_y_cmd_e_x = 0.0f;
static float eq_tau_z = 0.0f;
static float k_tau_z_psi = 0.0f;
static float k_tau_z_w_z = 0.0f;
static float eq_f_z = 0.0f;
static float k_f_z_p_z = 0.0f;
static float k_f_z_v_z = 0.0f;
static float k_f_z_e_z = 0.0f;


Print code that makes each gain a parameter in C, so that the actual value of each gain can be sent to the drone by the client just before each flight.

In [36]:
export_parameter_block(structure)

//                1234567890123456789012345 <-- max total length
//                group .name
PARAM_GROUP_START(ae483g)
PARAM_ADD(PARAM_FLOAT,   eq_tau_x_cmd,            &eq_tau_x_cmd)
PARAM_ADD(PARAM_FLOAT,   k_tau_x_cmd_p_y,         &k_tau_x_cmd_p_y)
PARAM_ADD(PARAM_FLOAT,   k_tau_x_cmd_phi,         &k_tau_x_cmd_phi)
PARAM_ADD(PARAM_FLOAT,   k_tau_x_cmd_v_y,         &k_tau_x_cmd_v_y)
PARAM_ADD(PARAM_FLOAT,   k_tau_x_cmd_w_x,         &k_tau_x_cmd_w_x)
PARAM_ADD(PARAM_FLOAT,   k_tau_x_cmd_tau_x,       &k_tau_x_cmd_tau_x)
PARAM_ADD(PARAM_FLOAT,   k_tau_x_cmd_e_y,         &k_tau_x_cmd_e_y)
PARAM_ADD(PARAM_FLOAT,   eq_tau_y_cmd,            &eq_tau_y_cmd)
PARAM_ADD(PARAM_FLOAT,   k_tau_y_cmd_p_x,         &k_tau_y_cmd_p_x)
PARAM_ADD(PARAM_FLOAT,   k_tau_y_cmd_theta,       &k_tau_y_cmd_theta)
PARAM_ADD(PARAM_FLOAT,   k_tau_y_cmd_v_x,         &k_tau_y_cmd_v_x)
PARAM_ADD(PARAM_FLOAT,   k_tau_y_cmd_w_y,         &k_tau_y_cmd_w_y)
PARAM_ADD(PARAM_FLOAT,   k_tau_y_cmd_tau_y,       &k_tau_y_cmd_ta

Print code that implements the controller in C.

In [37]:
export_controller(structure, s_with_des)

tau_x_cmd = eq_tau_x_cmd - (k_tau_x_cmd_p_y * (p_y - p_y_des) + k_tau_x_cmd_phi * phi + k_tau_x_cmd_v_y * v_y + k_tau_x_cmd_w_x * w_x + k_tau_x_cmd_tau_x * tau_x + k_tau_x_cmd_e_y * e_y);
tau_y_cmd = eq_tau_y_cmd - (k_tau_y_cmd_p_x * (p_x - p_x_des) + k_tau_y_cmd_theta * theta + k_tau_y_cmd_v_x * v_x + k_tau_y_cmd_w_y * w_y + k_tau_y_cmd_tau_y * tau_y + k_tau_y_cmd_e_x * e_x);
tau_z = eq_tau_z - (k_tau_z_psi * psi + k_tau_z_w_z * w_z);
f_z = eq_f_z - (k_f_z_p_z * (p_z - p_z_des) + k_f_z_v_z * v_z + k_f_z_e_z * e_z);


## 6. Gains

Q is the table in the paper. R is from Bryson's rule: the largest change in each input that keeps every motor command within [0, 65535] about hover.

`m_bryson` is the mass used for that headroom. For the empty-box hover test it stays at the bare-drone 0.0340 kg, so the gains keep the authority of the flown controller and only the hover thrust and linearization change with `m`. At 0.0390 kg the headroom is only about 1340 commands, and Bryson's rule would make the gains several times gentler. Set `m_bryson = m` for the final Task 5 gains once the hover test shows how much headroom there really is.

In [38]:
Q = np.diag([
    16.,   # p_x
    16.,   # p_y
    5.,    # p_z
    48.,   # psi
    64.,   # theta
    64.,   # phi
    128.,  # v_x
    128.,  # v_y
    2.,    # v_z
    1.,    # w_x
    1.,    # w_y
    1.,    # w_z
    1.,    # tau_x
    1.,    # tau_y
    16.,   # e_x
    16.,   # e_y
    16.,   # e_z
])

# Bryson's rule for R
m_bryson = 0.0340
m_hover_bryson = m_bryson * g / (4 * k_F)
dm = min(65535 - m_hover_bryson, m_hover_bryson)
u_max = np.abs(P) @ np.full(4, dm)
R = np.diag(1 / u_max**2)
print(f'R diagonal: {np.diag(R)}')

R diagonal: [2.82529935e+05 2.82529935e+05 2.48367863e+07 2.98422244e+02]


Motor commands at hover for this mass, and how much headroom is left.

In [39]:
m_hover = m * g / (4 * k_F)
print(f'mass {m * 1e3:.1f} g: hover command {m_hover:.0f} of 65535 ({m_hover / 65535:.1%}), '
      f'headroom {65535 - m_hover:.0f}')
print(f'max thrust / weight: {4 * k_F * 65535 / (m * g):.3f}')
if m_hover > 65535:
    print('WARNING: by this motor model the drone cannot hover at this mass')

mass 39.0 g: hover command 64055 of 65535 (97.7%), headroom 1480
max thrust / weight: 1.023


Find the gain matrix $K$ and check that the closed loop is stable.

In [40]:
K = lqr(A, B, Q, R)

F = A - B @ K
eigs = np.linalg.eigvals(F)
print(f'All eigenvalues of A - B K have negative real part: {np.all(eigs.real < 0)}')
print(f'Slowest: {np.max(eigs.real):.3f}, fastest: {np.min(eigs.real):.3f}')

All eigenvalues of A - B K have negative real part: True
Slowest: -0.456, fastest: -31.142


Show $K$ (formatted nicely).

In [41]:
K_str = np.array2string(K,
                        formatter={'float_kind': lambda x: f'{x:12.4f}'},
                        prefix='    ',
                        max_line_width=np.inf)

print(f'K = {K_str}')

K = [[      0.0000      -0.0208       0.0000      -0.0000       0.0000       0.0417       0.0000      -0.0250      -0.0000       0.0032       0.0000      -0.0000       3.4866      -0.0000       0.0000      -0.0075       0.0000]
     [      0.0208       0.0000      -0.0000       0.0000       0.0418      -0.0000       0.0250       0.0000      -0.0000      -0.0000       0.0032       0.0000      -0.0000       3.4405       0.0075       0.0000       0.0000]
     [      0.0000       0.0000      -0.0000       0.0014       0.0000      -0.0000       0.0000       0.0000      -0.0000      -0.0000       0.0000       0.0004      -0.0000       0.0000       0.0000       0.0000      -0.0000]
     [     -0.0000      -0.0000       0.3139      -0.0000      -0.0000       0.0000      -0.0000      -0.0000       0.1766       0.0000      -0.0000      -0.0000      -0.0000      -0.0000      -0.0000      -0.0000       0.2315]]


Write the gains. Set `gains_filename` in the flight script to this file.

### Hover thrust correction (temporary, no reflash)

The firmware turns `f_z` into motor commands with `k_F`, and `k_F` is about 14% low. In `task5_empty_run2.json` the drone hovered at a mean `f_z` of 0.3286 N in model units, not `m g` = 0.3826 N. When the feed-forward is `m g`, the drone overshoots on takeoff, and the `e_z` integrator has to wind up to cancel the extra 14% (that windup made the large height swings in `task5_path_empty_run1.json`).

So the exported `eq_f_z` is scaled by the measured ratio. Only the feed-forward changes: K is still designed at `m g`. Set `thrust_correction = 1.0` once `k_F` is corrected and the firmware is reflashed.

In [42]:
thrust_correction = 0.3286 / (0.0390 * 9.81)   # measured hover f_z / (m g), task5_empty_run2.json
i_eq_flight = [0., 0., 0., thrust_correction * m * g]
print(f'thrust_correction = {thrust_correction:.4f}: eq_f_z = {i_eq_flight[3]:.4f} N (was m g = {m * g:.4f} N)')

thrust_correction = 0.8589: eq_f_z = 0.3286 N (was m g = 0.3826 N)


In [43]:
export_gains(
    K,               # the gain matrix
    structure,       # the gain structure
    s,               # list of states as symbolic variables
    i,               # list of inputs as symbolic variables
    i_eq_flight,     # equilibrium inputs, with the measured hover thrust
    filename='gains_box_empty_ffz.json',
)

Wrote 21 gains to gains_box_empty_ffz.json at 2026-11-11T14:34:01
  eq_tau_x_cmd      =  0.00000000
  k_tau_x_cmd_p_y   = -0.02080725
  k_tau_x_cmd_phi   =  0.04170799
  k_tau_x_cmd_v_y   = -0.02500279
  k_tau_x_cmd_w_x   =  0.00315294
  k_tau_x_cmd_tau_x =  3.48658467
  k_tau_x_cmd_e_y   = -0.00752537
  eq_tau_y_cmd      =  0.00000000
  k_tau_y_cmd_p_x   =  0.02081099
  k_tau_y_cmd_theta =  0.04183004
  k_tau_y_cmd_v_x   =  0.02501314
  k_tau_y_cmd_w_y   =  0.00317367
  k_tau_y_cmd_tau_y =  3.44047059
  k_tau_y_cmd_e_x   =  0.00752537
  eq_tau_z          =  0.00000000
  k_tau_z_psi       =  0.00139019
  k_tau_z_w_z       =  0.00035732
  eq_f_z            =  0.32860000
  k_f_z_p_z         =  0.31390583
  k_f_z_v_z         =  0.17659719
  k_f_z_e_z         =  0.23154979
